# Kokoro TTS — Custom Voice Embedding Training

This notebook trains a new voice for [Kokoro-82M](https://huggingface.co/hexgrad/Kokoro-82M) by **optimising a voice embedding tensor** from reference audio recordings.

### How it works
Kokoro stores each voice as a `[511, 1, 256]` tensor saved as a `.pt` file.  
At inference time `pack[len(phonemes) - 1]` (shape `[1, 256]`) is selected and split:
- **`ref_s[:, :128]`** → fed to the Decoder via AdaIN layers (controls timbre / voice character)
- **`ref_s[:, 128:]`** → fed to the ProsodyPredictor (controls duration, F0, voicing)

Because both paths are differentiable with respect to the voice tensor, we can optimise it directly by comparing the mel spectrogram of the synthesised audio with reference recordings.

### What you need
- **5–60 minutes** of clean, single-speaker audio (WAV / MP3 / FLAC / OGG)
- Text transcripts **or** let Whisper auto-transcribe
- A free Colab GPU (T4 is fine)

### Output
A `your_voice_name.pt` file you can drop straight into your Kokoro pipeline:
```python
for gs, ps, audio in pipeline(text, voice='/path/to/your_voice.pt'):
    ...
```


In [ ]:
# @title ⚙️ Step 1 — Install dependencies { run: "auto" }
!pip install -q "kokoro>=0.9.4" openai-whisper soundfile torchaudio
!apt-get -qq -y install ffmpeg espeak-ng > /dev/null 2>&1
print('✅ Dependencies installed')

In [ ]:
import os, random, warnings
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.functional as AF
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import display, Audio
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('⚠️  No GPU detected — training will be slow. Go to Runtime → Change runtime type → T4 GPU.')
else:
    print(f'✅ Using device: {device} ({torch.cuda.get_device_name(0)})')

torch.manual_seed(42)
random.seed(42)

## Step 2 — Configuration
Edit the settings below before running the rest of the notebook.

In [ ]:
# @title ⚙️ Configuration

# ── Data ──────────────────────────────────────────────────────────────────
AUDIO_DIR        = '/content/voice_samples'  # @param {type:"string"}
# Set to None to auto-transcribe with Whisper; or point to a folder of .txt files
TRANSCRIPT_DIR   = None  # @param

# Language code: 'a' = American English, 'b' = British English
# Other supported: 'e' (Spanish), 'f' (French), 'h' (Hindi), 'i' (Italian),
#                  'p' (Portuguese), 'j' (Japanese), 'z' (Mandarin)
LANG_CODE        = 'a'   # @param ["a", "b", "e", "f", "h", "i", "p", "j", "z"]

# ── Output ────────────────────────────────────────────────────────────────
# Output voice name — follows Kokoro convention: {lang}{gender}_{name}
# e.g. 'af_myvoice', 'am_myvoice', 'bf_myvoice'
VOICE_NAME       = 'af_myvoice'  # @param {type:"string"}
OUTPUT_DIR       = '/content/output'  # @param {type:"string"}

# ── Warm-start voice ──────────────────────────────────────────────────────
# Starting from an existing voice speeds up convergence.
# Pick the built-in voice that sounds most similar to your target.
# Set to None to start from scratch (slower).
BASE_VOICE       = 'af_heart'   # @param {type:"string"}

# ── Training ──────────────────────────────────────────────────────────────
NUM_STEPS        = 400   # @param {type:"integer"}
LEARNING_RATE    = 5e-3  # @param {type:"number"}
GRAD_ACCUM_STEPS = 4     # @param {type:"integer"}
SMOOTHNESS_WEIGHT= 0.01  # @param {type:"number"}
SAVE_EVERY       = 50    # @param {type:"integer"}

# ── Audio preprocessing ───────────────────────────────────────────────────
MAX_CLIP_SECONDS = 8     # @param {type:"number"}
MIN_CLIP_SECONDS = 1     # @param {type:"number"}
TARGET_SR        = 24000  # Kokoro's native sample rate — do not change

os.makedirs(AUDIO_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✅ Configuration saved')

## Step 3 — (Optional) Mount Google Drive
Run this cell if your audio is stored in Google Drive. Skip otherwise.

In [ ]:
# @title 📂 Mount Google Drive (optional)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Drive mounted at /content/drive')
    print('   Update AUDIO_DIR above to point to your Drive folder, e.g.')
    print('   AUDIO_DIR = "/content/drive/MyDrive/voice_samples"')
except Exception:
    print('Not running in Colab or drive already mounted.')

## Step 4 — Upload reference audio

Run **one** of the two cells below:
- **Cell A** — upload files directly from your computer
- **Cell B** — skip upload (files already in `AUDIO_DIR` or Drive)

**Tips for best results**
- Use **clean, noise-free** recordings (studio or quiet room)
- All recordings must be **one speaker only**
- Aim for **10–60 minutes** total audio across many utterances
- Short clips (3–8 s) work better than one long file
- Varied content (different sentences) improves generalisation

In [ ]:
# @title 🎙️ Cell A — Upload audio files
from google.colab import files as colab_files

print(f'Uploading to {AUDIO_DIR} ...')
uploaded = colab_files.upload()
for fname, data in uploaded.items():
    dest = os.path.join(AUDIO_DIR, fname)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved: {dest}')

audio_files = [
    os.path.join(AUDIO_DIR, f) for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac'))
]
print(f'\n✅ Found {len(audio_files)} audio file(s) in {AUDIO_DIR}')

In [ ]:
# @title 📁 Cell B — Scan existing directory (skip upload)
audio_files = [
    os.path.join(AUDIO_DIR, f) for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac'))
]
print(f'✅ Found {len(audio_files)} audio file(s) in {AUDIO_DIR}')
if not audio_files:
    print('⚠️  No audio files found — check AUDIO_DIR or run Cell A to upload.')

## Step 5 — Preprocess audio
Resamples everything to 24 kHz mono and splits long recordings into short clips.

In [ ]:
PROCESSED_DIR = '/content/processed_clips'
os.makedirs(PROCESSED_DIR, exist_ok=True)

processed_files = []
clip_samples   = int(MAX_CLIP_SECONDS * TARGET_SR)
min_samples    = int(MIN_CLIP_SECONDS * TARGET_SR)

for fpath in audio_files:
    try:
        waveform, sr = torchaudio.load(fpath)
    except Exception as e:
        print(f'  ⚠️  Could not load {os.path.basename(fpath)}: {e}')
        continue

    # Mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)

    # Resample
    if sr != TARGET_SR:
        waveform = AF.resample(waveform, sr, TARGET_SR)

    # Peak-normalise
    peak = waveform.abs().max()
    if peak > 0:
        waveform = waveform / peak * 0.95

    audio_1d = waveform.squeeze(0)
    stem     = os.path.splitext(os.path.basename(fpath))[0]

    # Split into clips
    clips = [
        audio_1d[i : i + clip_samples]
        for i in range(0, len(audio_1d), clip_samples)
    ]

    for idx, clip in enumerate(clips):
        if len(clip) < min_samples:
            continue
        out_path = os.path.join(PROCESSED_DIR, f'{stem}_{idx:04d}.wav')
        sf.write(out_path, clip.numpy(), TARGET_SR)
        processed_files.append(out_path)

print(f'✅ Created {len(processed_files)} clip(s) from {len(audio_files)} file(s)')
total_secs = len(processed_files) * MAX_CLIP_SECONDS
print(f'   Total audio: ~{total_secs/60:.1f} minutes')

## Step 6 — Transcribe with Whisper
Skipped automatically for clips that already have a matching `.txt` file in `TRANSCRIPT_DIR`.

In [ ]:
import whisper

# Whisper model size: 'tiny', 'base', 'small', 'medium', 'large'
# 'base' is a good balance of speed and accuracy for English.
WHISPER_MODEL = 'base'

print(f'Loading Whisper ({WHISPER_MODEL}) ...')
whisper_model = whisper.load_model(WHISPER_MODEL)
print('✅ Whisper loaded')

transcripts = {}   # clip_path -> transcript_text
whisper_lang = 'en' if LANG_CODE in ('a', 'b') else None

for fpath in processed_files:
    stem = os.path.splitext(os.path.basename(fpath))[0]

    # Check for pre-existing transcript
    if TRANSCRIPT_DIR:
        txt_path = os.path.join(TRANSCRIPT_DIR, stem + '.txt')
        if os.path.exists(txt_path):
            with open(txt_path, 'r', encoding='utf-8') as f:
                text = f.read().strip()
            if text:
                transcripts[fpath] = text
                continue

    # Auto-transcribe
    try:
        result = whisper_model.transcribe(
            fpath,
            language=whisper_lang,
            fp16=(device == 'cuda')
        )
        text = result['text'].strip()
        if text:
            transcripts[fpath] = text
    except Exception as e:
        print(f'  ⚠️  Transcription failed for {os.path.basename(fpath)}: {e}')

print(f'\n✅ Transcribed {len(transcripts)} clip(s)')

# Preview
for fpath, txt in list(transcripts.items())[:5]:
    print(f'  {os.path.basename(fpath)[:30]:30s}  "{txt[:70]}"')

## Step 7 — Load Kokoro model

In [ ]:
from kokoro import KModel, KPipeline

REPO_ID = 'hexgrad/Kokoro-82M'

print('Loading Kokoro model ...')
model = KModel(repo_id=REPO_ID).to(device).eval()

# Freeze ALL model weights — we only optimise the voice pack tensor
for param in model.parameters():
    param.requires_grad_(False)

# Pipeline (quiet — used only for G2P / phonemisation, not synthesis)
pipeline = KPipeline(lang_code=LANG_CODE, repo_id=REPO_ID, model=False)

n_params = sum(p.numel() for p in model.parameters())
print(f'✅ Kokoro loaded ({n_params:,} parameters, all frozen)')

## Step 8 — Build training dataset
Phonemises each transcript and pairs it with its audio clip.

In [ ]:
training_data = []
skipped = 0

for fpath, text in transcripts.items():
    try:
        waveform, sr = torchaudio.load(fpath)
        audio = waveform.squeeze(0)  # [T]
    except Exception as e:
        skipped += 1
        continue

    # Phonemise — use the quiet pipeline to get (graphemes, phonemes, None)
    try:
        chunks = list(pipeline(text, voice=None))
    except Exception as e:
        skipped += 1
        continue

    if not chunks:
        skipped += 1
        continue

    # Use first phoneme chunk (clips are short, so it should cover the whole clip)
    gs, ps, _ = chunks[0]
    ps = ps.strip()

    if not ps or len(ps) < 2 or len(ps) > 510:
        skipped += 1
        continue

    training_data.append({
        'audio':    audio,       # float32 tensor, 24 kHz
        'phonemes': ps,          # IPA phoneme string
        'ph_len':   len(ps),     # used to index into voice_pack
        'text':     gs,          # grapheme text (for logging)
    })

print(f'✅ Training samples: {len(training_data)}  (skipped {skipped})')
if len(training_data) == 0:
    raise RuntimeError('No training samples — check your audio files and transcripts.')

ph_lens = [d['ph_len'] for d in training_data]
print(f'   Phoneme lengths — min: {min(ph_lens)}, max: {max(ph_lens)}, mean: {np.mean(ph_lens):.0f}')

# Preview a few samples
print('\nSample entries:')
for item in random.sample(training_data, min(3, len(training_data))):
    print(f'  [{item["ph_len"]:3d} ph]  "{item["text"][:60]}"')

## Step 9 — Define training utilities

### Why a custom forward function?
`KModel.forward_with_tokens` is decorated with `@torch.no_grad()` to save memory at inference.  
For training we call the underlying sub-modules directly so autograd can compute gradients
**with respect to `ref_s`** (the voice embedding slice) through two differentiable paths:

| Path | Modules | Controls |
|------|---------|----------|
| `ref_s[:, :128]` | `Decoder` (AdaIN layers) | Timbre, voice character |
| `ref_s[:, 128:]` | `ProsodyPredictor` (DurationEncoder, F0/N networks) | Pitch, rhythm, voicing |

In [ ]:
# ── Differentiable synthesis forward ──────────────────────────────────────

def training_forward(model, phonemes: str, ref_s: torch.Tensor, speed: float = 1.0):
    """
    Synthesise audio from phonemes using ref_s, with full gradient support
    through the decoder and prosody predictor paths.

    Args:
        model:    KModel (all weights frozen)
        phonemes: IPA phoneme string (length 1–510)
        ref_s:    voice embedding slice, shape [1, 256]
        speed:    speech rate multiplier

    Returns:
        audio: waveform tensor [T] at 24 kHz, with grad_fn attached
    """
    dev = model.device

    # Phoneme → token IDs (same logic as KModel.forward)
    input_ids = [
        model.vocab[p] for p in phonemes if p in model.vocab
    ]
    if not input_ids:
        raise ValueError(f'No vocabulary matches for phonemes: {phonemes!r}')

    input_ids    = torch.LongTensor([[0, *input_ids, 0]]).to(dev)
    input_lengths = torch.full((1,), input_ids.shape[-1], dtype=torch.long, device=dev)
    text_mask    = torch.arange(input_lengths.max()).unsqueeze(0).expand(1, -1).type_as(input_lengths)
    text_mask    = torch.gt(text_mask + 1, input_lengths.unsqueeze(1)).to(dev)

    # ── Fixed encoder (no grad — fast path) ──
    with torch.no_grad():
        bert_dur = model.bert(input_ids, attention_mask=(~text_mask).int())
        d_en     = model.bert_encoder(bert_dur).transpose(-1, -2)
        t_en     = model.text_encoder(input_ids, input_lengths, text_mask)

    # ── Prosody style (differentiable w.r.t. ref_s[:, 128:]) ──
    s = ref_s[:, 128:]

    d        = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)
    x, _     = model.predictor.lstm(d)
    duration = model.predictor.duration_proj(x)
    duration = torch.sigmoid(duration).sum(axis=-1) / speed

    # Hard alignment — discrete rounding breaks the grad chain here,
    # but grads still flow through decoder and F0/N paths below.
    with torch.no_grad():
        pred_dur = torch.round(duration).clamp(min=1).long().squeeze()
        indices  = torch.repeat_interleave(
            torch.arange(input_ids.shape[1], device=dev), pred_dur)
        pred_aln_trg = torch.zeros(
            (input_ids.shape[1], indices.shape[0]), device=dev)
        pred_aln_trg[indices, torch.arange(indices.shape[0])] = 1
        pred_aln_trg = pred_aln_trg.unsqueeze(0)

    # ── F0 / noise prediction (differentiable w.r.t. ref_s[:, 128:]) ──
    en               = d.transpose(-1, -2) @ pred_aln_trg
    F0_pred, N_pred  = model.predictor.F0Ntrain(en, s)

    # ── Decoder (differentiable w.r.t. ref_s[:, :128]) ──
    asr   = t_en @ pred_aln_trg
    audio = model.decoder(asr, F0_pred, N_pred, ref_s[:, :128]).squeeze()

    return audio


# ── Mel transform (created once on device) ────────────────────────────────

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate = TARGET_SR,
    n_fft       = 1024,
    hop_length  = 256,
    win_length  = 1024,
    n_mels      = 80,
    f_min       = 0,
    f_max       = 8000,
).to(device)


def log_mel(audio: torch.Tensor) -> torch.Tensor:
    """Return log-mel spectrogram, shape [n_mels, T]."""
    mel = mel_transform(audio.unsqueeze(0).to(device)).squeeze(0)  # [80, T]
    return torch.log(mel.clamp(min=1e-5))


def voice_loss(gen_audio: torch.Tensor, ref_audio: torch.Tensor) -> torch.Tensor:
    """
    Compute a length-robust mel spectrogram loss between generated and reference audio.

    Three components:
      1. Mean spectral envelope  (timbre — completely length-invariant)
      2. Spectral std deviation  (dynamic range character)
      3. Frame-level overlap     (optional, weighted low — handles length mismatch)
    """
    gen_mel = log_mel(gen_audio)   # [80, T_gen]
    ref_mel = log_mel(ref_audio)   # [80, T_ref]

    # 1. Global mean — captures timbre without needing alignment
    loss = F.l1_loss(gen_mel.mean(-1), ref_mel.mean(-1))

    # 2. Global std — captures expressiveness / dynamics
    loss = loss + 0.5 * F.l1_loss(gen_mel.std(-1), ref_mel.std(-1))

    # 3. Overlapping frames — fine-grained spectral matching
    min_len = min(gen_mel.shape[-1], ref_mel.shape[-1])
    if min_len > 8:
        loss = loss + 0.2 * F.l1_loss(
            gen_mel[:, :min_len], ref_mel[:, :min_len]
        )

    return loss


def smoothness_reg(pack: torch.Tensor) -> torch.Tensor:
    """Encourage neighbouring embedding vectors to be similar."""
    return torch.mean((pack[1:] - pack[:-1]).pow(2))


print('✅ Training utilities defined')

## Step 10 — Initialise voice pack

The voice pack is a learnable parameter of shape **`[511, 1, 256]`**.

- **Warm start** (recommended): copy an existing built-in voice → faster convergence
- **Cold start**: random initialisation → slower but unbiased

In [ ]:
from torch import nn

if BASE_VOICE:
    print(f'Warm-starting from built-in voice: {BASE_VOICE}')
    # Load base voice via a temporary loud pipeline
    _tmp = KPipeline(lang_code=LANG_CODE, repo_id=REPO_ID, model=model)
    base_pack = _tmp.load_voice(BASE_VOICE).to(device)   # [511, 1, 256]
    del _tmp
    voice_pack = nn.Parameter(base_pack.clone().detach())
    print(f'  Initialised from {BASE_VOICE}, shape: {voice_pack.shape}')
else:
    print('Cold-starting from small random values')
    voice_pack = nn.Parameter(torch.randn(511, 1, 256, device=device) * 0.05)
    print(f'  Initialised randomly, shape: {voice_pack.shape}')

optimizer = torch.optim.Adam([voice_pack], lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_STEPS, eta_min=LEARNING_RATE * 0.01
)

trainable = voice_pack.numel()
print(f'\n✅ Learnable parameters: {trainable:,} ({trainable * 4 / 1024:.0f} KB)')

## Step 11 — Train

The training loop:
1. Randomly samples `GRAD_ACCUM_STEPS` clips per step
2. Synthesises audio using the current `voice_pack[ph_len - 1]` embedding
3. Computes mel spectrogram loss against the reference clip
4. Adds a smoothness regulariser across the 511 embedding vectors
5. Backpropagates **only into `voice_pack`** — the Kokoro model stays frozen

Run this cell and monitor the loss. It should drop steadily.  
Good results are typically seen after 200–400 steps.

In [ ]:
from tqdm.notebook import tqdm

losses       = []
best_loss    = float('inf')
best_pack    = voice_pack.detach().clone()

pbar = tqdm(range(1, NUM_STEPS + 1), desc='Training')

for step in pbar:
    optimizer.zero_grad()
    step_loss = 0.0
    valid     = 0

    # ── Mini-batch (gradient accumulation) ────────────────────────────────
    batch = random.sample(training_data, min(GRAD_ACCUM_STEPS, len(training_data)))

    for item in batch:
        phonemes  = item['phonemes']
        ref_audio = item['audio'].to(device)
        ph_len    = item['ph_len']

        # Select the embedding slice for this phoneme-length bucket
        ref_s = voice_pack[ph_len - 1]   # [1, 256]

        try:
            gen_audio = training_forward(model, phonemes, ref_s)
            loss      = voice_loss(gen_audio, ref_audio) / GRAD_ACCUM_STEPS
            loss.backward()
            step_loss += loss.item()
            valid     += 1
        except Exception as e:
            # Skip bad samples silently; warn only if every sample fails
            pass

    if valid == 0:
        pbar.set_postfix({'loss': 'all_failed'})
        continue

    # ── Smoothness regularisation ──────────────────────────────────────────
    smooth = SMOOTHNESS_WEIGHT * smoothness_reg(voice_pack)
    smooth.backward()
    step_loss += smooth.item()

    # ── Update voice pack ──────────────────────────────────────────────────
    nn.utils.clip_grad_norm_([voice_pack], max_norm=1.0)
    optimizer.step()
    scheduler.step()

    losses.append(step_loss)

    if step_loss < best_loss:
        best_loss = step_loss
        best_pack = voice_pack.detach().clone()

    pbar.set_postfix({'loss': f'{step_loss:.4f}', 'best': f'{best_loss:.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

    # ── Periodic checkpoint ────────────────────────────────────────────────
    if step % SAVE_EVERY == 0:
        ckpt_path = os.path.join(OUTPUT_DIR, f'{VOICE_NAME}_step{step:04d}.pt')
        torch.save(voice_pack.detach().cpu(), ckpt_path)

print(f'\n✅ Training complete — best loss: {best_loss:.4f}')

In [ ]:
# @title 📉 Plot loss curve
plt.figure(figsize=(10, 4))
plt.plot(losses, linewidth=0.8, label='step loss')

if len(losses) >= 20:
    window = max(1, len(losses) // 20)
    smooth_losses = np.convolve(losses, np.ones(window) / window, mode='valid')
    plt.plot(range(window - 1, len(losses)), smooth_losses,
             linewidth=2, label=f'smoothed (w={window})')

plt.xlabel('Step')
plt.ylabel('Loss')
plt.title(f'Voice Embedding Training Loss — {VOICE_NAME}')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'{VOICE_NAME}_loss.png'), dpi=150)
plt.show()
print(f'Plot saved to {OUTPUT_DIR}')

## Step 12 — Listen and evaluate

Synthesise a few test sentences with both the **best checkpoint** and the **final** voice pack
so you can compare and choose.

In [ ]:
# @title 🔊 Evaluate — listen to synthesised samples

TEST_SENTENCES = [
    'Hello! This is my new custom voice trained with Kokoro.',
    'The quick brown fox jumps over the lazy dog.',
    'Speech synthesis has come a long way in recent years.',
    'I hope this voice sounds natural and expressive to you.',
]

# Use a loud pipeline for synthesis — pass the model we already have
eval_pipeline = KPipeline(lang_code=LANG_CODE, repo_id=REPO_ID, model=model)

def synth_with_pack(pack, text, label):
    print(f'\n── {label} ── "{text[:60]}"')
    for gs, ps, audio in eval_pipeline(text, voice=pack.to(device)):
        if audio is not None:
            display(Audio(data=audio.numpy(), rate=TARGET_SR))
            break

for sentence in TEST_SENTENCES:
    synth_with_pack(best_pack,             sentence, 'BEST checkpoint')
    synth_with_pack(voice_pack.detach(),   sentence, 'Final checkpoint')

In [ ]:
# @title 🆚 Compare against base voice
# Quick A/B comparison between your new voice and the base voice you started from.

COMPARE_TEXT = 'The human voice is the most natural instrument in the world.'

print(f'Text: "{COMPARE_TEXT}"\n')

# New voice (best)
print('New voice (best checkpoint):')
for gs, ps, audio in eval_pipeline(COMPARE_TEXT, voice=best_pack.to(device)):
    if audio is not None:
        display(Audio(data=audio.numpy(), rate=TARGET_SR))
        break

# Base voice for comparison
if BASE_VOICE:
    print(f'\nBase voice ({BASE_VOICE}):')
    for gs, ps, audio in eval_pipeline(COMPARE_TEXT, voice=BASE_VOICE):
        if audio is not None:
            display(Audio(data=audio.numpy(), rate=TARGET_SR))
            break

## Step 13 — Export voice

Save your voice pack as a `.pt` file.  
The final shape must be `[511, 1, 256]` — Kokoro expects this exact format.

In [ ]:
# Choose which checkpoint to export: 'best' or 'final'
EXPORT_CHECKPOINT = 'best'   # @param ["best", "final"]

export_pack = best_pack if EXPORT_CHECKPOINT == 'best' else voice_pack.detach().cpu()

# Verify shape
assert export_pack.shape == (511, 1, 256), (
    f'Unexpected shape {export_pack.shape}, expected (511, 1, 256)'
)

out_path = os.path.join(OUTPUT_DIR, f'{VOICE_NAME}.pt')
torch.save(export_pack.cpu(), out_path)

size_kb = os.path.getsize(out_path) / 1024
print(f'✅ Voice saved to: {out_path}')
print(f'   Shape : {tuple(export_pack.shape)}')
print(f'   Size  : {size_kb:.0f} KB')
print()
print('Usage in your pipeline:')
print(f'    pipeline(text, voice="{out_path}")')

In [ ]:
# @title 📥 Download voice file
try:
    from google.colab import files as colab_files
    print(f'Downloading {VOICE_NAME}.pt ...')
    colab_files.download(out_path)
except Exception:
    print(f'Not in Colab, or download blocked. File is at: {out_path}')

## Appendix A — Quick test at any point during training

Run this cell at any time to hear the current voice pack without interrupting the training loop.

In [ ]:
# @title 🎧 Quick test (run anytime)
QUICK_TEST_TEXT = 'Hello, this is a quick test of the voice in progress.'  # @param {type:"string"}

_test_pipeline = KPipeline(lang_code=LANG_CODE, repo_id=REPO_ID, model=model)
current_pack   = voice_pack.detach().to(device)

for gs, ps, audio in _test_pipeline(QUICK_TEST_TEXT, voice=current_pack):
    if audio is not None:
        print(f'Phonemes: {ps}')
        display(Audio(data=audio.numpy(), rate=TARGET_SR))
        break

## Appendix B — Resume training from a checkpoint

If the Colab session times out, use this cell to load the last saved checkpoint and continue.

In [ ]:
# @title ↩️ Resume from checkpoint
RESUME_PATH = '/content/output/af_myvoice_step0200.pt'  # @param {type:"string"}

if os.path.exists(RESUME_PATH):
    loaded = torch.load(RESUME_PATH, map_location=device, weights_only=True)
    assert loaded.shape == (511, 1, 256), f'Bad shape: {loaded.shape}'
    voice_pack = nn.Parameter(loaded.detach().clone())
    optimizer  = torch.optim.Adam([voice_pack], lr=LEARNING_RATE)
    print(f'✅ Resumed from {RESUME_PATH}')
    print('   Re-run the training cell to continue.')
else:
    print(f'⚠️  File not found: {RESUME_PATH}')

## Appendix C — Blend voices

Mix your new voice with one or more built-in voices by averaging the embedding tensors.

In [ ]:
# @title 🎨 Blend new voice with existing voices
BLEND_VOICES = 'af_heart,af_bella'   # @param {type:"string"} Built-in voices to blend with
BLEND_WEIGHT = 0.5                   # @param {type:"slider", min:0, max:1, step:0.05}
# Weight of YOUR voice (1.0 = pure new voice, 0.0 = pure built-in blend)

_blend_pipeline = KPipeline(lang_code=LANG_CODE, repo_id=REPO_ID, model=model)
builtin_packs   = [_blend_pipeline.load_voice(v.strip()) for v in BLEND_VOICES.split(',')]
builtin_mean    = torch.stack(builtin_packs).mean(0).to(device)

blended_pack = (
    BLEND_WEIGHT       * best_pack.to(device) +
    (1 - BLEND_WEIGHT) * builtin_mean
).detach()

BLEND_TEST_TEXT = 'This is the blended voice result.'  # @param {type:"string"}
print(f'Blended voice ({BLEND_WEIGHT:.0%} new + {1-BLEND_WEIGHT:.0%} built-in):')
for gs, ps, audio in _blend_pipeline(BLEND_TEST_TEXT, voice=blended_pack):
    if audio is not None:
        display(Audio(data=audio.numpy(), rate=TARGET_SR))
        break

# Save blend
blend_out = os.path.join(OUTPUT_DIR, f'{VOICE_NAME}_blend{int(BLEND_WEIGHT*100)}.pt')
torch.save(blended_pack.cpu(), blend_out)
print(f'\n✅ Blend saved to: {blend_out}')